# 04 GDP と生活水準

**GDP を最大にする配分と、生活水準を最大にする配分は一致するか**を調べます（RQ2・RQ6）。
GDP・1 人あたり GDP・健康指数・人的資本指数・生活水準を区別します。

> **補遺トピック G1・教育用の架空モデル**です。国名・経済指標・投資効果の係数はすべて架空で、現実の国家の GDP・経済成長率・政策効果を予測するものではありません。実際の政策判断に使わないでください。

## 準備：実験環境（最初に必ず実行）

G1 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                     # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g1-economic-growth-optimization"   # リポジトリの中の G1 フォルダ（ルートが G1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g1_root(p):
    return (Path(p) / "economic_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_g1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G1 のルート
        if is_g1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g1_upload")
    for hit in glob.glob("/content/g1_upload/**/economic_model.py", recursive=True):
        if is_g1_root(Path(hit).parent):
            return Path(hit).parent
    return None


G1_ROOT = find_g1_root()
if G1_ROOT is None:
    raise RuntimeError("G1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G1_ROOT)
if str(G1_ROOT) not in sys.path:
    sys.path.insert(0, str(G1_ROOT))
if not (G1_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。economic_model.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import economic_model as em
import simulation as sim
import allocation_strategies as al
import optimization as op
import evaluation as ev
import growth_charts as gc

gc.setup_style()                        # 文字を大きくするなら gc.setup_style(1.3)
PARAMS = em.load_economy()              # 架空の国家のパラメータ
SCENARIOS = al.load_scenarios()         # 比較用の配分シナリオ
LABELS = {sc["id"]: sc["label"] for sc in SCENARIOS}
YEARS = PARAMS["simulation_years"]
STEP = 0.1                              # 最適化の配分の刻み（10%）
M = al.units_count(STEP)

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", gc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


def alloc_cols():
    return [(s, em.SECTOR_LABELS[s], "{:.0%}") for s in em.SECTORS]


print("G1_ROOT:", G1_ROOT)
print(PARAMS["country"], "｜", YEARS, "年、年間予算", PARAMS["annual_investment_budget"], "（架空の単位）")
print("シナリオ:", [sc["label"] for sc in SCENARIOS])

## 1. 目的関数別の最適配分（全探索）

In [ ]:
#@title 目的関数別の最適配分
best = {}
for objective in op.OBJECTIVES:
    r = op.exhaustive(op.Evaluator(PARAMS, objective), M)
    best[objective] = r["shares"]
rows = []
for objective, shares in best.items():
    recs = sim.simulate(shares, PARAMS)
    last = recs[-1]
    rows.append({"name": op.OBJECTIVE_LABELS[objective], **shares, "gdp": last["gdp"],
                 "gdp_pc": last["gdp_per_capita_index"], "health": last["health_index"],
                 "hc": last["human_capital_index"], "ls": last["living_standard"]})
table(rows, [("name", "optimized for", "{}")] + alloc_cols()
      + [("gdp", "GDP", "{:.1f}"), ("gdp_pc", "GDP/cap idx", "{:.3f}"), ("health", "health idx", "{:.3f}"),
         ("hc", "human cap. idx", "{:.3f}"), ("ls", "living std", "{:.3f}")])
show(gc.plot_allocation_bars([(op.OBJECTIVE_LABELS[o].split(" (")[0].replace(" ", "\n", 1), s) for o, s in best.items()],
                             "Optimal allocation for each objective (exhaustive search)"), "04_alloc_by_objective")

## 2. GDP と生活水準の散布図（すべての配分候補）

1 点が 1 つの配分（刻み 10%、286 通り）。★ と ◆ は、それぞれの最大値の配分です。右上の端（Pareto フロンティア）では、片方を上げるともう片方が下がります。

In [ ]:
cands = op.enumerate_allocations(M)
points = []
for u in cands:
    recs = sim.simulate(al.shares_from_units(u, M), PARAMS)
    points.append((recs[-1]["gdp"], recs[-1]["living_standard"], al.shares_from_units(u, M)))
def point_of(shares):
    r = sim.simulate(shares, PARAMS)[-1]
    return (r["gdp"], r["living_standard"])
highlights = {"max final GDP": point_of(best["final_gdp"]),
              "max living standard": point_of(best["final_living_standard"]),
              "equal allocation": point_of(al.equal_allocation())}
show(gc.plot_tradeoff_scatter(points, highlights, color_by="healthcare"), "04_scatter")

## 3. GDP 重視と生活水準重視の比較

各指標を「初年度 = 1」の指数にそろえて比べます。

In [ ]:
#@title GDP 重視と生活水準重視
keys = [("gdp", "GDP index"), ("gdp_pc", "GDP per capita idx"), ("health", "Health idx"),
        ("hc", "Human capital idx"), ("ls", "Living standard")]
pick = {"GDP focus": rows[0], "Living-standard focus": rows[2], "Weighted": rows[4]}
vals = {name: [(r["gdp"] / PARAMS["initial_gdp"] if k == "gdp" else r[k]) for k, _l in keys] for name, r in pick.items()}
show(gc.plot_grouped_bars([l for _k, l in keys], list(pick), vals,
                          f"GDP focus vs living-standard focus, year {YEARS} (model)", "index (year 0 = 1)",
                          colors={"GDP focus": "#3d3c39", "Living-standard focus": "#6250d6", "Weighted": "#86b6ef"}),
     "04_focus")

## 4. 学生向け TODO：生活水準の重みを変える

In [ ]:
# TODO 1：生活水準の重みを変更する（合計 1）
P2 = em.with_changes(PARAMS, living_standard_weight_gdp_per_capita=0.6,
                     living_standard_weight_health=0.2, living_standard_weight_human_capital=0.2)
# TODO 2：最適配分を再計算する
for name, p in [("元の重み", PARAMS), ("変えた重み", P2)]:
    r = op.exhaustive(op.Evaluator(p, "final_living_standard"), M)
    print(f"{name}: 生活水準を最大にする配分 {al.format_shares(r['shares'])}  値 {r['objective']:.3f}")
# weighted の gdp_weight（GDP 指数の重み）を変える
for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = op.exhaustive(op.Evaluator(PARAMS, "weighted", gdp_weight=w), M)
    print(f"gdp_weight {w:.2f}: {al.format_shares(r['shares'])}")
# TODO 3：結果を考察する

## 考察（チームで記入）

- **TODO**：GDP 最大化と生活水準最大化の配分は一致したか。違うなら、どの分野の配分が違ったか（RQ2）
- **TODO**：すべての指標を同時に最大にする配分はあったか。トレードオフはどこにあるか（RQ6）
- **TODO**：生活水準の重みはモデル上の選択である。重みを変えると結論はどう変わるか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 キュー、第6章 単純挿入ソート）
- G1 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界）
- データ：`data/baseline_economy.json`、`data/investment_scenarios.json`（この教材のための**架空のデータ**）